In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import date,datetime

date_f = datetime.today().strftime('%Y%m%d %H-%M-%S')

spark = SparkSession.builder\
    .appName('test_code')\
        .getOrCreate()



##Extracting the data

df = spark.read\
    .format('csv')\
    .option('header',True)\
    .option('inferschema',True)\
    .load('s3://priya-de-project-2026/raw/incoming/customers.csv')




In [0]:
##Transforming the data


#Remove the spaces
extra_spaces = df.withColumn('removed_spaces', trim(col('emp_name')))

#convert to upper
convert_upper_case = extra_spaces.withColumn('department_upper',upper(col('department')))

#convert to lower
convert_lower = convert_upper_case.withColumn('email_lower',lower(col('email')))

#extract 3 characters 
character_length = convert_lower.withColumn('extracted_characters',substring(col('removed_spaces'),1,3))

#combining

combine = character_length.withColumn('combines',concat(col('removed_spaces'), lit('-'),col('department_upper') ))

#extract username

extract = combine.withColumn('username_email',split(col('email'),'@')[0])

#special_characters
new_number = extract.withColumn('new_number', regexp_replace(col('phone'),'[^0-9]',''))


#dropping unwanted columns
dropping = new_number.drop('emp_name','department','email','phone')


In [0]:
##Loading the data

dropping.write\
    .format('parquet')\
    .mode('overwrite')\
    .save(f'/Volumes/pri_catalog/pri_schema/pri_volume/pyspark_string_functions_{date_f}')